# NovaAI QLoRA fine-tuning

Qwen/Qwen2.5-1.5B-Instruct · NF4 · PEFT LoRA · T4

NovaAI is synthetic/fictional. Evaluation tests paraphrase recall of trained facts, not unseen facts. Only adapters are trained. No results are prefilled.

Choose **Runtime → Change runtime type → T4 GPU**, then **Run all**. The training cell actually trains when you run it.

## 1. Bootstrap (also start here after a restart)

Clones the full repository if absent and installs the pinned stack into an isolated environment. Stages run in fresh processes, so no installation restart is normally needed. After any kernel restart rerun Section 1, Section 2, then your unfinished stage. A deleted Colab VM loses local files; download artifacts before ending the session.

Until these local changes are published, upload/extract the **updated full repository** to `/content/LoRA-Fine-Tuning-Pipeline`. GitHub cloning retrieves the published version; uploading only this notebook cannot supply unpublished source changes.

In [ ]:
import importlib.util
import os
from pathlib import Path
import subprocess
import sys

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
if not (3, 10) <= sys.version_info[:2] <= (3, 12):
    raise RuntimeError("The pinned stack requires Python 3.10–3.12.")
if IN_COLAB:
    root = Path("/content/LoRA-Fine-Tuning-Pipeline")
    if not root.exists():
        subprocess.run(["git", "clone",
            "https://github.com/Azoqoz/LoRA-Fine-Tuning-Pipeline.git", str(root)], check=True)
else:
    root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "src/pipeline.py").is_file()), None)
    if root is None:
        raise FileNotFoundError("Run from the updated repository.")
os.chdir(root)
required = ["requirements.txt", "configs/training_config.yaml",
            "src/__init__.py", "src/runtime.py", "src/pipeline.py",
            "src/dataset_utils.py", "src/inference.py", "src/evaluation.py"]
missing = [p for p in required if not (root / p).is_file()]
if missing:
    raise FileNotFoundError(f"Missing updated files: {missing}. Use the updated full repository.")
print("Colab:", IN_COLAB, "| repository:", root)
environment = root / ".venv-colab"
python = environment / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
if not python.is_file():
    subprocess.run([sys.executable, "-m", "pip", "install", "virtualenv==20.34.0"], check=True)
    subprocess.run([sys.executable, "-m", "virtualenv", str(environment)], check=True)
torch_pin = next(line for line in (root / "requirements.txt").read_text().splitlines()
                 if line.startswith("torch=="))
subprocess.run([str(python), "-m", "pip", "install", torch_pin,
                "--index-url", "https://download.pytorch.org/whl/cu126"], check=True)
subprocess.run([str(python), "-m", "pip", "install", "-r", "requirements.txt"], check=True)
subprocess.run([str(python), "-m", "pip", "check"], check=True)
print("Continue to Section 2. No notebook kernel restart required.")


## Optional: restore missing datasets

Normally skip: cloned data is already present. Set the flag only for missing files. Existing files are never overwritten.

In [ ]:
from pathlib import Path
UPLOAD_MISSING_DATA = False
root = Path("/content/LoRA-Fine-Tuning-Pipeline") if Path("/content/LoRA-Fine-Tuning-Pipeline").is_dir() else Path.cwd()
names = ["train.jsonl", "validation.jsonl", "test.jsonl", "dataset_summary.json"]
missing = [name for name in names if not (root / "data" / name).is_file()]
if missing and UPLOAD_MISSING_DATA:
    from google.colab import files
    uploaded = files.upload()
    for name in missing:
        if name not in uploaded:
            raise FileNotFoundError(f"Missing upload: {name}")
        (root / "data").mkdir(exist_ok=True)
        (root / "data" / name).write_bytes(uploaded[name])
elif missing:
    raise FileNotFoundError(f"Missing data: {missing}. Enable the optional upload flag.")


## 2. GPU, dataset and QLoRA preflight

Checks CUDA, prints the GPU, executes a small NF4 layer, validates all splits and complete-chat lengths, resolves the model revision, and loads Qwen in 4-bit to verify NF4, double quantization, compute dtype and target modules. No training. The process exits to release GPU memory.

In [ ]:
import os
from pathlib import Path
import subprocess

root = Path("/content/LoRA-Fine-Tuning-Pipeline") if Path("/content/LoRA-Fine-Tuning-Pipeline").is_dir() else Path.cwd()
python = root / ".venv-colab" / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
if not python.is_file():
    raise RuntimeError("Rerun Section 1 (bootstrap) first.")
subprocess.run([str(python), "-m", "src.pipeline", "preflight"], cwd=root, check=True)


## 3. Normal FP16 baseline

A future execution archives the existing unprovenanced CSV byte-for-byte before generating all 70 answers with metadata. Matching completed predictions are reused after restarts; incompatible provenance is rejected. Process exit frees the full base model.

In [ ]:
import os
from pathlib import Path
import subprocess

root = Path("/content/LoRA-Fine-Tuning-Pipeline") if Path("/content/LoRA-Fine-Tuning-Pipeline").is_dir() else Path.cwd()
python = root / ".venv-colab" / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
if not python.is_file():
    raise RuntimeError("Rerun Section 1 (bootstrap) first.")
subprocess.run([str(python), "-m", "src.pipeline", "baseline"], cwd=root, check=True)


## 4. Train QLoRA and save

Actual training begins here using YAML settings. Epoch checkpoints resume within the verified experiment. Saves adapter, tokenizer, trainer state/checkpoints and an adapter hash manifest. Completed runs are verified and skipped on rerun. For a new experiment use distinct output paths.

In [ ]:
import os
from pathlib import Path
import subprocess

root = Path("/content/LoRA-Fine-Tuning-Pipeline") if Path("/content/LoRA-Fine-Tuning-Pipeline").is_dir() else Path.cwd()
python = root / ".venv-colab" / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
if not python.is_file():
    raise RuntimeError("Rerun Section 1 (bootstrap) first.")
subprocess.run([str(python), "-m", "src.pipeline", "train"], cwd=root, check=True)


## 5. Reload saved adapter and predict

Always reloads the saved tokenizer, exact base revision and adapter from disk in a new process after hash validation. Generates on the same test prompts. No model globals are reused.

In [ ]:
import os
from pathlib import Path
import subprocess

root = Path("/content/LoRA-Fine-Tuning-Pipeline") if Path("/content/LoRA-Fine-Tuning-Pipeline").is_dir() else Path.cwd()
python = root / ".venv-colab" / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
if not python.is_file():
    raise RuntimeError("Rerun Section 1 (bootstrap) first.")
subprocess.run([str(python), "-m", "src.pipeline", "predict"], cwd=root, check=True)


## 6. Compare and export metrics

Rejects missing or mismatched provenance and changed CSVs. Retains lexical metrics and category summaries. Expected-token coverage is not full factual correctness. FP16 baseline versus 4-bit tuned inference includes quantization effects.

In [ ]:
import os
from pathlib import Path
import subprocess

root = Path("/content/LoRA-Fine-Tuning-Pipeline") if Path("/content/LoRA-Fine-Tuning-Pipeline").is_dir() else Path.cwd()
python = root / ".venv-colab" / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
if not python.is_file():
    raise RuntimeError("Rerun Section 1 (bootstrap) first.")
subprocess.run([str(python), "-m", "src.pipeline", "evaluate"], cwd=root, check=True)


## 7. Download artifacts

Includes adapter, checkpoints, config, metadata and results; the archive can be large. Run early to preserve available artifacts after an interruption.

In [ ]:
from pathlib import Path
import zipfile

root = Path("/content/LoRA-Fine-Tuning-Pipeline") if Path("/content/LoRA-Fine-Tuning-Pipeline").is_dir() else Path.cwd()
archive = root / "novaai-run.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as output:
    for directory in ("artifacts", "results"):
        for path in sorted((root / directory).rglob("*")):
            if path.is_file():
                output.write(path, path.relative_to(root))
print("Saved", archive)
try:
    from google.colab import files
except ImportError:
    print("Download from your file browser.")
else:
    files.download(str(archive))
